# Experimento D.1 — Alineación y métricas de poses propias vs. oficiales

Comparamos las poses calculadas por nuestro pipeline SfM (Experimento C) con las poses oficiales  
incluidas en el dataset mip-NeRF 360 (`data/{scene}/sparse/0/`).

**Método de alineación:** Similaridad de Umeyama (escala + rotación + traslación global).  
**Métricas por cámara:** error de traslación (distancia euclidiana de centros alineados) y  
error de rotación (distancia geodésica en SO(3)).

> `garden` no tiene poses oficiales en el dataset → se omite y se documenta.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pycolmap
from pathlib import Path
from datetime import datetime

print("pycolmap:", pycolmap.__version__)


In [ ]:
EXP_D_DIR   = Path('./expD')
EXP_C_DIR   = Path('./expC')
DATA_DIR    = Path('./data')

RESULTS_DIR = EXP_D_DIR / 'results'
PLOTS_DIR   = EXP_D_DIR / 'plots'
for d in [RESULTS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Config del pipeline propio a usar para comparación
CONFIG_FOR_D = 'Cam_OpenCV'

# Escenas — garden se omite por no tener poses oficiales
SCENES_D = ['bonsai', 'bicycle', 'counter']

# Verificar disponibilidad de datos
print("=== Verificación de datos disponibles ===\n")
rec_df = None
rec_csv = EXP_C_DIR / 'results' / 'reconstructions_for_D.csv'
if rec_csv.exists():
    rec_df = pd.read_csv(rec_csv)
    print(f"reconstructions_for_D.csv cargado ({len(rec_df)} filas)")
else:
    print("reconstructions_for_D.csv no encontrado — usando paths por defecto")

for scene in ['garden'] + SCENES_D:
    official = DATA_DIR / scene / 'sparse' / '0'
    status = "✓ oficial" if official.exists() else "✗ SIN poses oficiales"
    print(f"  {scene:10s}: {status}  ({official})")


## Funciones: carga de poses COLMAP

In [ ]:
def load_colmap_poses(sparse_path):
    """
    Carga poses desde una reconstrucción COLMAP sparse.
    Retorna dict: image_name -> {'center': (3,), 'R': (3,3), 't': (3,)}
    donde center = posición de la cámara en espacio mundo.
    Soporta pycolmap >= 0.5 y versiones anteriores.
    """
    reco = pycolmap.Reconstruction(str(sparse_path))
    poses = {}

    for img_id, img in reco.images.items():
        # Obtener R y t (cam_from_world: p_cam = R @ p_world + t)
        try:
            R = np.array(img.cam_from_world.rotation.matrix())
            t = np.array(img.cam_from_world.translation)
        except AttributeError:
            # API anterior: qvec=[qw,qx,qy,qz], tvec
            from scipy.spatial.transform import Rotation as ScipyRot
            q = img.qvec  # [qw, qx, qy, qz]
            R = ScipyRot.from_quat([q[1], q[2], q[3], q[0]]).as_matrix()
            t = np.array(img.tvec)

        # Centro de cámara en espacio mundo: C = -R^T @ t
        center = -R.T @ t
        poses[img.name] = {'center': center, 'R': R, 't': t}

    return poses, reco.num_reg_images()


def get_student_sparse_path(scene, config=CONFIG_FOR_D):
    """Busca el path de la reconstrucción propia para una escena y config."""
    # Primero busca en reconstructions_for_D.csv
    if rec_df is not None:
        row = rec_df[(rec_df['scene'] == scene) & (rec_df['config'] == config)]
        if len(row) > 0 and pd.notna(row.iloc[0]['sparse_path']):
            p = Path(row.iloc[0]['sparse_path'])
            if p.exists():
                return p

    # Fallback: path convencional
    fallback = EXP_C_DIR / 'sparse' / scene / config / '0'
    return fallback if fallback.exists() else None


## Alineación: transformación de similaridad (Umeyama)

In [ ]:
def umeyama_alignment(src_pts, dst_pts):
    """
    Estima la transformación de similaridad que minimiza ||dst - (s*R@src + t)||^2.
    
    Parámetros:
        src_pts, dst_pts: arrays (N, 3) de puntos correspondientes.
    
    Retorna:
        s   — escala escalar
        R   — matriz de rotación (3, 3)
        t   — traslación (3,)
    
    Uso: dst_aligned = s * (R @ src.T).T + t
    """
    n      = len(src_pts)
    mu_s   = src_pts.mean(axis=0)
    mu_d   = dst_pts.mean(axis=0)
    src_c  = src_pts - mu_s
    dst_c  = dst_pts - mu_d

    var_s  = np.mean(np.sum(src_c ** 2, axis=1))
    H      = (dst_c.T @ src_c) / n
    U, S, Vt = np.linalg.svd(H)

    # Corrige reflexiones
    d      = np.sign(np.linalg.det(U @ Vt))
    D      = np.diag([1.0, 1.0, d])

    R      = U @ D @ Vt
    s      = np.trace(np.diag(S) @ D) / var_s
    t      = mu_d - s * R @ mu_s

    return s, R, t


def apply_sim_transform(pts, s, R, t):
    """Aplica dst = s * R @ src + t a un array (N, 3)."""
    return (s * (R @ pts.T)).T + t


def rotation_error_deg(R1, R2):
    """
    Distancia geodésica entre dos rotaciones en grados.
    error = arccos( (trace(R1^T @ R2) - 1) / 2 )
    """
    R_rel   = R1.T @ R2
    cos_val = np.clip((np.trace(R_rel) - 1.0) / 2.0, -1.0, 1.0)
    return float(np.degrees(np.arccos(cos_val)))


## Comparación de poses por escena

In [ ]:
def compare_poses_scene(scene):
    """
    Alinea y compara poses propias vs oficiales para una escena.
    Retorna dict con métricas agregadas y per-imagen.
    """
    print(f"\n{'='*50}")
    print(f"Escena: {scene}")
    print('='*50)

    # ── Cargar poses ────────────────────────────────────────────────────────
    student_path  = get_student_sparse_path(scene)
    official_path = DATA_DIR / scene / 'sparse' / '0'

    if student_path is None or not student_path.exists():
        print(f"  [!] Sin reconstrucción propia para {scene} ({CONFIG_FOR_D})")
        return None
    if not official_path.exists():
        print(f"  [!] Sin poses oficiales para {scene} — omitiendo")
        return None

    student_poses,  n_stu = load_colmap_poses(student_path)
    official_poses, n_off = load_colmap_poses(official_path)

    print(f"  Propias:   {n_stu} imágenes  ({student_path})")
    print(f"  Oficiales: {n_off} imágenes  ({official_path})")

    # ── Imágenes comunes ────────────────────────────────────────────────────
    common = sorted(set(student_poses.keys()) & set(official_poses.keys()))
    print(f"  Comunes:   {len(common)}")

    if len(common) < 4:
        print("  [!] Demasiado pocas imágenes comunes para alinear de forma fiable")
        return None

    src_pts = np.array([student_poses[n]['center']  for n in common])
    dst_pts = np.array([official_poses[n]['center'] for n in common])

    # ── Alineación Umeyama ──────────────────────────────────────────────────
    s, R_align, t_align = umeyama_alignment(src_pts, dst_pts)
    src_aligned = apply_sim_transform(src_pts, s, R_align, t_align)

    print(f"  Escala de alineación: {s:.6f}")

    # ── Errores por cámara ──────────────────────────────────────────────────
    per_img = []
    for i, name in enumerate(common):
        # Error de traslación: distancia euclidiana entre centros alineados
        t_err = float(np.linalg.norm(src_aligned[i] - dst_pts[i]))

        # Error de rotación: distancia geodésica en SO(3)
        # En espacio oficial: R_student_aligned ≈ R_student @ R_align^T
        R_stu  = student_poses[name]['R']
        R_off  = official_poses[name]['R']
        R_aligned = R_stu @ R_align.T
        r_err  = rotation_error_deg(R_aligned, R_off)

        per_img.append({'image': name, 't_error': t_err, 'r_error_deg': r_err})

    df_per = pd.DataFrame(per_img)
    t_errs = df_per['t_error'].values
    r_errs = df_per['r_error_deg'].values

    summary = {
        'scene'                      : scene,
        'config'                     : CONFIG_FOR_D,
        'n_student_images'           : n_stu,
        'n_official_images'          : n_off,
        'n_common_images'            : len(common),
        'alignment_scale'            : float(s),
        'mean_t_error'               : float(np.mean(t_errs)),
        'median_t_error'             : float(np.median(t_errs)),
        'std_t_error'                : float(np.std(t_errs)),
        'p90_t_error'                : float(np.percentile(t_errs, 90)),
        'mean_r_error_deg'           : float(np.mean(r_errs)),
        'median_r_error_deg'         : float(np.median(r_errs)),
        'std_r_error_deg'            : float(np.std(r_errs)),
        'p90_r_error_deg'            : float(np.percentile(r_errs, 90)),
        'fraction_t_under_0.1'       : float(np.mean(t_errs < 0.1)),
        'fraction_r_under_5deg'      : float(np.mean(r_errs < 5.0)),
        'student_sparse_path'        : str(student_path),
        'official_sparse_path'       : str(official_path),
        'timestamp'                  : datetime.now().isoformat(),
    }

    print(f"\n  Error traslación — mean: {summary['mean_t_error']:.4f} | "
          f"median: {summary['median_t_error']:.4f} | "
          f"std: {summary['std_t_error']:.4f} | "
          f"p90: {summary['p90_t_error']:.4f}")
    print(f"  Error rotación   — mean: {summary['mean_r_error_deg']:.2f}° | "
          f"median: {summary['median_r_error_deg']:.2f}° | "
          f"std: {summary['std_r_error_deg']:.2f}° | "
          f"p90: {summary['p90_r_error_deg']:.2f}°")
    print(f"  Cámaras con t < 0.1:  {summary['fraction_t_under_0.1']*100:.1f}%")
    print(f"  Cámaras con r < 5°:   {summary['fraction_r_under_5deg']*100:.1f}%")

    return summary, df_per


# Correr para todas las escenas disponibles
all_summaries  = []
all_per_image  = {}

for scene in SCENES_D:
    result = compare_poses_scene(scene)
    if result is not None:
        summary, df_per = result
        all_summaries.append(summary)
        all_per_image[scene] = df_per

df_summary = pd.DataFrame(all_summaries)
print("\n\n=== Resumen global ===")
cols_s = ['scene', 'n_common_images', 'alignment_scale',
          'mean_t_error', 'median_t_error',
          'mean_r_error_deg', 'median_r_error_deg']
print(df_summary[cols_s].to_string(index=False))

# Garden — documentar ausencia
print("\n[garden] No tiene poses oficiales en el dataset (sparse/0/ ausente).")
print("         No es posible realizar la comparación para esta escena.")


## Visualización de distribuciones de error

In [ ]:
if not all_per_image:
    print("Sin datos para graficar")
else:
    scenes_avail = list(all_per_image.keys())
    n_scenes = len(scenes_avail)

    fig, axes = plt.subplots(2, n_scenes, figsize=(5 * n_scenes, 8))
    if n_scenes == 1:
        axes = axes.reshape(2, 1)
    fig.suptitle('D.1 — Distribución de errores de poses propias vs. oficiales',
                 fontsize=13, fontweight='bold')

    for col, scene in enumerate(scenes_avail):
        df_p = all_per_image[scene]
        t_errs = df_p['t_error'].values
        r_errs = df_p['r_error_deg'].values

        # Histograma traslación
        ax_t = axes[0, col]
        ax_t.hist(t_errs, bins=30, color='steelblue', edgecolor='white', alpha=0.85)
        ax_t.axvline(np.mean(t_errs),   color='red',    lw=1.5, ls='--', label=f'media={np.mean(t_errs):.3f}')
        ax_t.axvline(np.median(t_errs), color='orange', lw=1.5, ls=':',  label=f'mediana={np.median(t_errs):.3f}')
        ax_t.set_title(f'{scene} — Error traslación')
        ax_t.set_xlabel('distancia euclidiana (unidades escena)')
        ax_t.set_ylabel('# cámaras')
        ax_t.legend(fontsize=8)

        # Histograma rotación
        ax_r = axes[1, col]
        ax_r.hist(r_errs, bins=30, color='coral', edgecolor='white', alpha=0.85)
        ax_r.axvline(np.mean(r_errs),   color='red',    lw=1.5, ls='--', label=f'media={np.mean(r_errs):.2f}°')
        ax_r.axvline(np.median(r_errs), color='orange', lw=1.5, ls=':',  label=f'mediana={np.median(r_errs):.2f}°')
        ax_r.set_title(f'{scene} — Error rotación')
        ax_r.set_xlabel('distancia geodésica (grados)')
        ax_r.set_ylabel('# cámaras')
        ax_r.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'pose_error_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Guardado en {PLOTS_DIR / 'pose_error_distributions.png'}")


In [ ]:
# CDF acumulada de errores — útil para comparar escenas
if all_per_image:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle('D.1 — CDF de errores por escena', fontsize=12, fontweight='bold')

    colors = plt.cm.tab10(np.linspace(0, 0.6, len(all_per_image)))

    for (scene, df_p), color in zip(all_per_image.items(), colors):
        t_sorted = np.sort(df_p['t_error'].values)
        r_sorted = np.sort(df_p['r_error_deg'].values)
        cdf = np.arange(1, len(t_sorted) + 1) / len(t_sorted)

        ax1.plot(t_sorted, cdf, label=scene, color=color, lw=2)
        ax2.plot(r_sorted, cdf, label=scene, color=color, lw=2)

    ax1.set_xlabel('Error traslación (distancia euclidiana)'); ax1.set_ylabel('CDF')
    ax1.set_title('CDF — Error de traslación'); ax1.legend(); ax1.grid(alpha=0.3)

    ax2.set_xlabel('Error rotación (grados)'); ax2.set_ylabel('CDF')
    ax2.set_title('CDF — Error de rotación'); ax2.legend(); ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'pose_error_cdf.png', dpi=150, bbox_inches='tight')
    plt.show()


## Guardar resultados

In [ ]:
# Guardar resumen
if not df_summary.empty:
    df_summary.to_csv(RESULTS_DIR / 'pose_comparison_summary.csv', index=False)
    print(f"Resumen guardado: {RESULTS_DIR / 'pose_comparison_summary.csv'}")

# Guardar errores por imagen para cada escena
for scene, df_p in all_per_image.items():
    out = RESULTS_DIR / f'pose_errors_per_image_{scene}.csv'
    df_p.to_csv(out, index=False)
    print(f"Errores por imagen ({scene}): {out}")


## Discusión

### Elección de métricas
- **Error de traslación** (distancia euclidiana de centros alineados): intuitivo y en unidades de la escena.  
  Sin embargo, depende de la escala — por eso se normaliza mediante la alineación de similaridad de Umeyama.
- **Error de rotación** (distancia geodésica en SO(3)): mide el ángulo mínimo de rotación que separa las dos orientaciones.  
  Es independiente de la representación (no sufre de singularidades como los ángulos de Euler).

### Escala de alineación
Un factor de escala ≠ 1 indica que nuestra reconstrucción difiere en escala absoluta respecto al dataset oficial,  
lo cual es esperable dado que SfM recupera geometría hasta una ambigüedad de escala sin referencia métrica.

### Interpretación de los errores
- Errores de traslación pequeños y error de rotación < 5° indican poses bien recuperadas.
- Errores altos en pocas cámaras (cola derecha de la distribución) suelen corresponder a imágenes difíciles  
  (movimiento borroso, oclusión, poca superposición) o a errores de matching.

### garden — ausencia de poses oficiales
El dataset mip-NeRF 360 no incluye la carpeta `sparse/0/` para la escena `garden` en nuestra versión descargada.  
No es posible realizar la comparación cuantitativa para esta escena.
